# STEAM PROJECT

In [0]:
filepath = "s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json"

In [0]:
df = spark.read.format('json').load(filepath, multiline=True)

In [0]:
df.count()

55691

In [0]:
df.limit(5).toPandas()

,data,id
0,"{'appid': 10, 'categories': ['Multi-player', '...",10
1,"{'appid': 1000000, 'categories': ['Single-play...",1000000
2,"{'appid': 1000010, 'categories': ['Single-play...",1000010
3,"{'appid': 1000030, 'categories': ['Multi-playe...",1000030
4,"{'appid': 1000040, 'categories': ['Single-play...",1000040


In [0]:
df.printSchema()

root
 |-- data: struct (nullable = true)
 |    |-- appid: long (nullable = true)
 |    |-- categories: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- ccu: long (nullable = true)
 |    |-- developer: string (nullable = true)
 |    |-- discount: string (nullable = true)
 |    |-- genre: string (nullable = true)
 |    |-- header_image: string (nullable = true)
 |    |-- initialprice: string (nullable = true)
 |    |-- languages: string (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- negative: long (nullable = true)
 |    |-- owners: string (nullable = true)
 |    |-- platforms: struct (nullable = true)
 |    |    |-- linux: boolean (nullable = true)
 |    |    |-- mac: boolean (nullable = true)
 |    |    |-- windows: boolean (nullable = true)
 |    |-- positive: long (nullable = true)
 |    |-- price: string (nullable = true)
 |    |-- publisher: string (nullable = true)
 |    |-- release_date: string (nullable = true)
 |    |-

In [0]:
df.columns

['data', 'id']

=> 2 main columns: data and id.

22 sub-columns in data. categories, platforms and tags have sub-levels

# EDA

## Column 'data'

In [0]:
from pyspark.sql import functions as F

df_data = (
    df.select(
        F.col("data.appid").alias("appid"),
        F.col("data.ccu").alias("ccu"),
        F.col("data.developer").alias("developer_name"),
        F.col("data.discount").alias("discount"),
        F.col("data.genre").alias("genre"),
        F.col("data.header_image").alias("header_image"),
        F.col("data.initialprice").alias("initialprice"),
        F.col("data.price").alias("price"),
        F.col("data.languages").alias("languages_text"),
        F.col("data.name").alias("name"),
        F.col("data.positive").alias("positive"),
        F.col("data.negative").alias("negative"),
        F.col("data.owners").alias("owners_text"),
        F.col("data.publisher").alias("publisher_name"),
        F.col("data.release_date").alias("release_date"),
        F.col("data.required_age").alias("required_age"),
        F.col("data.short_description").alias("short_description"),
        F.col("data.type").alias("type"),
        F.col("data.website").alias("website")
    )
)

#display(df_data)

=> developer_name,genre, owners_text and publisher_names seem to need investigating

In [0]:
df.select("data").distinct().count()

55691

In [0]:
distinct_counts = df_data.select([
    F.countDistinct(F.col(c)).alias(c) for c in df_data.columns
])

distinct_counts.show(truncate=False)

+-----+----+--------------+--------+-----+------------+------------+-----+--------------+-----+--------+--------+-----------+--------------+------------+------------+-----------------+----+-------+
|appid|ccu |developer_name|discount|genre|header_image|initialprice|price|languages_text|name |positive|negative|owners_text|publisher_name|release_date|required_age|short_description|type|website|
+-----+----+--------------+--------+-----+------------+------------+-----+--------------+-----+--------+--------+-----------+--------------+------------+------------+-----------------+----+-------+
|55691|1113|34773         |64      |1832 |55691       |203         |385  |8623          |55430|4529    |2224    |13         |29966         |4011        |21          |55332            |2   |25457  |
+-----+----+--------------+--------+-----+------------+------------+-----+--------------+-----+--------+--------+-----------+--------------+------------+------------+-----------------+----+-------+



### Column genre

In [0]:
display(df_data.select("genre").distinct().orderBy(F.col("genre")))

genre
""
Accounting
"Accounting, Animation & Modeling, Audio Production, Design & Illustration, Education, Photo Editing, Software Training, Utilities, Video Production, Web Publishing"
"Accounting, Animation & Modeling, Audio Production, Design & Illustration, Education, Photo Editing, Software Training, Utilities, Video Production, Web Publishing, Game Development"
"Accounting, Education, Software Training, Utilities, Early Access"
"Accounting, Utilities"
Action
"Action, Adventure"
"Action, Adventure, Casual"
"Action, Adventure, Casual, Early Access"


=> genre needs to be exploded and put in a dimension table

## Column languages_text

In [0]:
df_data.select("languages_text").distinct().orderBy(F.col("languages_text")).show()

+--------------------+
|      languages_text|
+--------------------+
|                    |
|              Arabic|
|               Czech|
|Czech, Danish, Du...|
|Czech, Dutch, Eng...|
|Czech, Dutch, Eng...|
|Czech, English, F...|
|Czech, English, F...|
|Czech, English, F...|
|Czech, English, F...|
|Czech, English, F...|
|Danish, Dutch, En...|
|Danish, Dutch, En...|
|Danish, Dutch, En...|
|Danish, Dutch, En...|
|Danish, Dutch, En...|
|Dutch, English, F...|
|Dutch, English, F...|
|Dutch, English, F...|
|             English|
+--------------------+
only showing top 20 rows


=> Will need explode...

### Columns POSITIVE AND NEGATIVE

In [0]:
display(df_data.select('positive', 'negative'))

positive,negative
201215,5199
27,5
4032,646
1575,115
0,1
1018,462
18,6
50,34
6,0
32,12


=> will need to create a "total_reviews" and "positive_rate" (=positive/postive + negative) measures

### Column OWNERS

In [0]:
df_data.select("owners_text").distinct().orderBy(F.col("owners_text")).show()

+--------------------+
|         owners_text|
+--------------------+
|         0 .. 20,000|
|1,000,000 .. 2,00...|
|10,000,000 .. 20,...|
|  100,000 .. 200,000|
|2,000,000 .. 5,00...|
|    20,000 .. 50,000|
|20,000,000 .. 50,...|
|  200,000 .. 500,000|
|200,000,000 .. 50...|
|5,000,000 .. 10,0...|
|   50,000 .. 100,000|
|50,000,000 .. 100...|
|500,000 .. 1,000,000|
+--------------------+



=> owners_text is an already "binned" quantity of owners. I keep this as is but can be used a filters. However I will need to make them nicer (replace ..) and to extract the first number so that I can sort them.

## Column PUBLISHER

In [0]:
df_data.select("publisher_name").distinct().show(50, truncate=False)

+----------------------------------------------+
|publisher_name                                |
+----------------------------------------------+
|Atari                                         |
|Headup, WhisperGames                          |
|太阳的后羿                                    |
|Dupeyron                                      |
|CHARON                                        |
|Rose City Games                               |
|Akamaru Game                                  |
|The Quantum Astrophysicists Guild, OneGuyGames|
|Free Initiative Games                         |
|Ed Del Castillo                               |
|Virtual Journey Studio                        |
|Anti-Ded GameDev                              |
|Double Mojo                                   |
|MagicHouse                                    |
|Meng Games                                    |
|LAN - GAMES LTD                               |
|SZEINER                                       |
|Polyblock Studio        

=> there seem to be joint publisher that will need to be divided

### RELEASE_DATE

In [0]:
df_data.select("release_date").distinct().show(50, truncate=False)

+------------+
|release_date|
+------------+
|2019/08/19  |
|2009/03/4   |
|2019/01/26  |
|2020/06/24  |
|2019/06/12  |
|2021/12/10  |
|2019/01/1   |
|2019/10/14  |
|2020/07/28  |
|2019/02/23  |
|2019/03/28  |
|2019/04/26  |
|2019/11/18  |
|2019/08/15  |
|2022/10/24  |
|2020/05/20  |
|2021/02/12  |
|2021/09/21  |
|2021/11/30  |
|2019/10/4   |
|2020/07/10  |
|2022/08/27  |
|2020/05/12  |
|2019/09/7   |
|2021/10/20  |
|2020/11/2   |
|2019/07/28  |
|2021/03/16  |
|2020/01/27  |
|2021/01/13  |
|2021/02/16  |
|2020/08/28  |
|2019/07/24  |
|2019/11/12  |
|2019/05/4   |
|2019/05     |
|2021/04/11  |
|2014/02/14  |
|2019/04/12  |
|2019/09/25  |
|2020/03/9   |
|2019/06/18  |
|2020/10/17  |
|2019/10/24  |
|2021/03/12  |
|2019/07/25  |
|2020/01/24  |
|2019/06/9   |
|2020/10/20  |
|2020/02/21  |
+------------+
only showing top 50 rows


=> _Date format will have to be normalized_

## Column REQUIRED_AGE

In [0]:
df_data.groupBy("required_age") \
      .agg(F.count("*").alias("rows")) \
      .orderBy(F.desc("rows")) \
      .show(200, truncate=False)

+------------+-----+
|required_age|rows |
+------------+-----+
|0           |55030|
|15          |264  |
|18          |223  |
|16          |38   |
|17          |38   |
|12          |32   |
|13          |26   |
|14          |10   |
|10          |7    |
|6           |4    |
|180         |4    |
|8           |3    |
|3           |3    |
|7           |2    |
|21+         |1    |
|9           |1    |
|35          |1    |
|7+          |1    |
|MA 15+      |1    |
|5           |1    |
|20          |1    |
+------------+-----+



=> MA 15+ to be changed and 180 ???

## Column TYPE

In [0]:
df_data.groupBy("type") \
      .agg(F.count("*").alias("rows")) \
      .orderBy(F.desc("rows")) \
      .show(200, truncate=False)

+--------+-----+
|type    |rows |
+--------+-----+
|game    |55690|
|hardware|1    |
+--------+-----+



=> Usuless, drop

In [0]:
df_data.groupBy("initialprice").count().orderBy(F.desc("count")).show(500, truncate=False)

+------------+-----+
|initialprice|count|
+------------+-----+
|0           |7780 |
|499         |6518 |
|999         |6336 |
|99          |5441 |
|199         |4285 |
|299         |3738 |
|1499        |3027 |
|1999        |2992 |
|399         |2591 |
|699         |1964 |
|599         |1926 |
|799         |1631 |
|2499        |936  |
|2999        |906  |
|1299        |851  |
|1199        |850  |
|899         |756  |
|1099        |460  |
|3999        |447  |
|1799        |297  |
|1399        |272  |
|1599        |228  |
|4999        |200  |
|5999        |194  |
|1699        |189  |
|3499        |161  |
|1899        |150  |
|4499        |47   |
|90          |34   |
|6999        |27   |
|9999        |24   |
|500         |20   |
|19999       |18   |
|100         |17   |
|5499        |15   |
|7999        |14   |
|2199        |12   |
|249         |12   |
|1000        |10   |
|1500        |10   |
|149         |9    |
|300         |9    |
|420         |8    |
|200         |7    |
|6499        

In [0]:
df_data.groupBy("price").count().orderBy(F.desc("count")).show(500, truncate=False)

+-----+-----+
|price|count|
+-----+-----+
|0    |7780 |
|499  |6250 |
|999  |6126 |
|99   |5243 |
|199  |4193 |
|299  |3639 |
|1499 |2878 |
|1999 |2820 |
|399  |2530 |
|699  |1856 |
|599  |1854 |
|799  |1580 |
|2999 |865  |
|2499 |862  |
|1199 |822  |
|1299 |796  |
|899  |753  |
|1099 |430  |
|3999 |428  |
|49   |303  |
|1799 |286  |
|1399 |276  |
|1599 |230  |
|4999 |190  |
|5999 |187  |
|1699 |185  |
|3499 |155  |
|1899 |141  |
|249  |99   |
|59   |97   |
|149  |73   |
|51   |51   |
|349  |45   |
|4499 |44   |
|449  |41   |
|79   |40   |
|749  |40   |
|119  |40   |
|69   |38   |
|74   |34   |
|209  |34   |
|90   |32   |
|374  |29   |
|239  |27   |
|50   |25   |
|89   |25   |
|6999 |25   |
|159  |25   |
|9999 |24   |
|179  |23   |
|279  |22   |
|124  |21   |
|54   |21   |
|28   |20   |
|359  |20   |
|500  |20   |
|139  |19   |
|37   |19   |
|479  |18   |
|19999|17   |
|100  |16   |
|5499 |15   |
|1249 |14   |
|219  |14   |
|974  |13   |
|7999 |13   |
|649  |13   |
|224  |12   |
|174  

In [0]:
df_data.groupBy("discount").count().orderBy(F.desc("count")).show(500, truncate=False)

+--------+-----+
|discount|count|
+--------+-----+
|0       |53173|
|50      |350  |
|90      |239  |
|80      |228  |
|75      |223  |
|40      |164  |
|51      |146  |
|60      |137  |
|70      |137  |
|30      |131  |
|20      |112  |
|25      |85   |
|10      |53   |
|15      |45   |
|35      |39   |
|65      |37   |
|87      |36   |
|85      |34   |
|83      |26   |
|48      |25   |
|45      |25   |
|55      |24   |
|81      |23   |
|72      |22   |
|66      |20   |
|69      |20   |
|33      |16   |
|71      |14   |
|86      |13   |
|67      |11   |
|82      |8    |
|89      |7    |
|74      |6    |
|22      |6    |
|62      |5    |
|37      |5    |
|61      |4    |
|77      |4    |
|49      |3    |
|18      |3    |
|34      |3    |
|59      |2    |
|24      |2    |
|27      |2    |
|42      |2    |
|56      |2    |
|88      |2    |
|79      |1    |
|68      |1    |
|44      |1    |
|29      |1    |
|12      |1    |
|64      |1    |
|38      |1    |
|52      |1    |
|78      |1   

# CLEANING

## GENRE : explode and create bridge table

In [0]:
bridge_game_genre = (
    df_data
      .select("appid", F.explode(F.split(F.col("genre"), r"\s*,\s*")).alias("genre"))
      .filter(F.col("genre").isNotNull() & (F.col("genre") != ""))
      .dropDuplicates(["appid", "genre"])
)

bridge_game_genre.show(20, truncate=False)

+-------+---------------------+
|appid  |genre                |
+-------+---------------------+
|1021220|Adventure            |
|1039580|Casual               |
|1016770|Strategy             |
|1027530|Indie                |
|1029640|Early Access         |
|1032140|RPG                  |
|1037410|Adventure            |
|1006430|Indie                |
|1008870|Action               |
|1010860|Massively Multiplayer|
|1012120|Action               |
|1013000|Casual               |
|1013120|RPG                  |
|1023320|Free to Play         |
|1024490|Simulation           |
|1001450|Adventure            |
|1014180|Strategy             |
|1029750|Casual               |
|1009510|Action               |
|1022990|Early Access         |
+-------+---------------------+
only showing top 20 rows


## LANGUAGES : explode and create bridge table

In [0]:
languages_distinct = (
    df_data
    .select("languages_text")
    .where(F.col("languages_text").isNotNull() & (F.length(F.trim("languages_text")) > 0))
    .withColumn("language", F.explode_outer(F.split(F.col("languages_text"), "[,;]")))
    .withColumn("language", F.trim(F.col("language")))
    .where(F.col("language") != "")
    .dropDuplicates(["language"])
    .orderBy("language")
)

languages_distinct.show(200, truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------+
|languages_text                                                                                                                                                                                                                                                                                                                                                                                                                  |language                                                            |
+-----------------------

In [0]:
# 1) Flatten & normalize to base languages
_lang_clean = (
    df_data
    .select("appid", "languages_text")
    .where(F.col("languages_text").isNotNull() & (F.length(F.trim("languages_text")) > 0))
    # unify separators: turn newlines into commas
    .withColumn("txt", F.regexp_replace("languages_text", r"\r?\n", ","))
    # split on comma/semicolon
    .withColumn("token", F.explode_outer(F.split(F.col("txt"), r"[,;]")))
    .withColumn("token", F.trim(F.col("token")))
    .where(F.col("token") != "")
    # drop parentheticals like "(Full Audio)", "(text only)"
    .withColumn("token", F.regexp_replace(F.col("token"), r"\([^)]*\)", ""))
    # drop simple BBCode noise like [b]*[/b]
    .withColumn("token", F.regexp_replace(F.col("token"), r"\[/?[biu]\]|\*", ""))
    # remove country/region suffix after a hyphen, e.g. "Portuguese - Brazil" -> "Portuguese"
    .withColumn("base_language", F.regexp_replace(F.col("token"), r"\s*-\s*.*$", ""))
    # normalize spaces & casing
    .withColumn("base_language", F.trim(F.regexp_replace(F.col("base_language"), r"\s{2,}", " ")))
    .withColumn("base_language", F.initcap(F.lower(F.col("base_language"))))
    # drop obvious junk
    .where(~F.col("base_language").rlike("^(Not Supported|All With Full Audio Support)$"))
    .where(~F.col("base_language").startswith("#"))
)

# 2) See ALL distinct base languages (sorted)
languages_distinct = (
    _lang_clean
    .select("base_language")
    .dropDuplicates()
    .orderBy("base_language")
)

languages_distinct.show(500, truncate=False)

# 3) Slim bridge: appid ↔ base language
bridge_game_language = (
    _lang_clean
    .select("appid", F.col("base_language").alias("language"))
    .dropDuplicates(["appid", "language"])
)

# cleanup for null/blank values
bridge_game_language = bridge_game_language.filter(
    F.col("language").isNotNull() & (F.trim("language") != "")
)

bridge_game_language.show(20, truncate=False)

+---------------------+
|base_language        |
+---------------------+
|                     |
|Afrikaans            |
|Albanian             |
|Arabic               |
|Azerbaijani          |
|Bangla               |
|Basque               |
|Belarusian           |
|Bosnian              |
|Bulgarian            |
|Catalan              |
|Croatian             |
|Czech                |
|Danish               |
|Dari                 |
|Dutch                |
|English              |
|English Dutch English|
|Estonian             |
|Filipino             |
|Finnish              |
|French               |
|Galician             |
|Georgian             |
|German               |
|Greek                |
|Hebrew               |
|Hindi                |
|Hungarian            |
|Icelandic            |
|Indonesian           |
|Irish                |
|Italian              |
|Japanese             |
|Kannada              |
|Kazakh               |
|Korean               |
|Latvian              |
|Lithuanian     

## POSITIVE + NEGATIVE

In [0]:
fact_games = (
    df_data
        .withColumn("total_reviews", F.col("positive") + F.col("negative"))
        .withColumn(
            "positive_rate",
            F.when(
                (F.col("positive") + F.col("negative")) > 0,
                F.col("positive") / (F.col("positive") + F.col("negative"))
            ).otherwise(None)
        )
)

fact_games.show(1)

+-----+-----+--------------+--------+------+--------------------+------------+-----+--------------------+--------------+--------+--------+--------------------+--------------+------------+------------+--------------------+----+-------+-------------+------------------+
|appid|  ccu|developer_name|discount| genre|        header_image|initialprice|price|      languages_text|          name|positive|negative|         owners_text|publisher_name|release_date|required_age|   short_description|type|website|total_reviews|     positive_rate|
+-----+-----+--------------+--------+------+--------------------+------------+-----+--------------------+--------------+--------+--------+--------------------+--------------+------------+------------+--------------------+----+-------+-------------+------------------+
|   10|13990|         Valve|       0|Action|https://cdn.akama...|         999|  999|English, French, ...|Counter-Strike|  201215|    5199|10,000,000 .. 20,...|         Valve|   2000/11/1|         

## OWNERS

In [0]:
# improve the look of the ranges
fact_games = fact_games.withColumn(
    "owners_label",
    F.regexp_replace(
        #F.regexp_replace("owners_text", " .. ", "-"),
        F.regexp_replace("owners_text", r"\s\.\.\s", "-"),  # escape the dots
        ",",
        ""
    )
)

In [0]:
# extract the first number of each range in a new column in order to be able to sort (in visuals)
fact_games = fact_games.withColumn(
    "owners_min",
    F.split("owners_label", "-").getItem(0).cast("long")
)


display(fact_games.select("owners_label").distinct().orderBy(F.col("owners_min")))

owners_label
0-20000
20000-50000
50000-100000
100000-200000
200000-500000
500000-1000000
1000000-2000000
2000000-5000000
5000000-10000000
10000000-20000000


## PUBLISHERS

In [0]:
# Show 50 distinct publishers whose name contains a comma (for inspection)
publishers_with_comma = (
    df_data
        .where(F.col("publisher_name").isNotNull() & (F.instr("publisher_name", ",") > 0))
        .select(F.trim("publisher_name").alias("publisher_name"))
        .distinct()
        .orderBy("publisher_name")
        .limit(50)
)

publishers_with_comma.show(truncate=False)

+-------------------------------------------------------------------+
|publisher_name                                                     |
+-------------------------------------------------------------------+
|++Good Games, GameChanger Charity                                  |
|+Mpact Games, LLC.                                                 |
|1 Trait Danger, Matador Records                                    |
|10mg, Gesinimo Games                                               |
|13-lab, azimut team                                                |
|13-lab, azimuth team                                               |
|13AM Shipping Solutions, 13AM Games                                |
|18Light Game Ltd., CE-Asia, 樂磚 Joy Brick, Mayflower Entertainment|
|18Light Game Ltd., Cheese Games                                    |
|20 Below Games, Inc                                                |
|24Frame,inc.                                                       |
|2GT, Dexter.CO       

In [0]:
bridge_game_publisher = (
    pub
      .withColumn("publisher_raw", F.explode_outer(F.split("pub", r"\|")))
      .withColumn("publisher_raw", F.trim("publisher_raw"))

      # remove surrounding quotes and leading/trailing punctuation junk
      .withColumn("publisher_raw", F.regexp_replace("publisher_raw", r'^[\'"\.\+\-\s]+', ""))
      .withColumn("publisher_raw", F.regexp_replace("publisher_raw", r'[\'"\.\+\-\s]+$', ""))
      # collapse internal whitespace and stray quotes
      .withColumn("publisher_raw", F.regexp_replace("publisher_raw", r'\s+', " "))
      .withColumn("publisher_raw", F.regexp_replace("publisher_raw", r'["\']', ""))

      # keep only tokens that contain at least one letter
      .where(F.col("publisher_raw").rlike(r'.*[A-Za-z].*'))

      # drop suffix-only tokens like "LLC", "AB", ". Ltd.", etc.
      .where(~F.col("publisher_raw").rlike(
          r'(?i)^(inc\.?|incorporated|llc|l\.?l\.?c\.?|ltd\.?|limited|gmbh|sas|s\.?a\.?s\.?|sarl|s\.?a\.?r\.?l\.?|sa|ab|bv|oy|k\.?k\.?|co\.?\s*ltd\.?|pty\s*ltd|pvt\.?\s*ltd\.?)$'
      ))

      # final canonical case: Title Case (case-insensitive de-dupe)
      .withColumn("publisher", F.initcap(F.lower(F.col("publisher_raw"))))

      .select("appid", "publisher")
      .dropDuplicates()
)

# sanity check
bridge_game_publisher.select("publisher").distinct().orderBy("publisher").show(100, truncate=False)



+-----------------------+
|publisher              |
+-----------------------+
|((no-end-parens Studio |
|0 Deer Soft Partnership|
|000 Little Birds       |
|007berd                |
|011 Games              |
|021workshop            |
|0cube                  |
|0dark&nerdy            |
|0o0                    |
|0up Games              |
|1 Flame Studio         |
|1 Trait Danger         |
|1-reality              |
|1.5 Hp Games           |
|10 Chambers            |
|10 Games               |
|100 Plus Games Llc     |
|100 Stones Interactive |
|1001 Studio            |
|100hr Games            |
|10101 Software         |
|1013133 Simulations    |
|101xp                  |
|1047 Games             |
|1081games              |
|109 Below              |
|10f                    |
|10ft Games             |
|10mg                   |
|10ravens S.r.o         |
|10space                |
|10th Reality           |
|10tons Ltd             |
|10x10 Room             |
|11 Bit Studios         |
|11114444777

# Building the fact table

In [0]:
# reformat the release dates
fact_games = (
    fact_games
        .withColumn(
            "release_norm",
            F.when((F.col("release_date") == "") | F.col("release_date").isNull(), None)
             .when(F.length("release_date") == 7,  # yyyy/MM
                   F.concat_ws("/", F.col("release_date"), F.lit("01")))
             .when(F.length("release_date") == 9,  # yyyy/MM/d  (single-digit day)
                   F.concat(
                       F.substring("release_date", 1, 8),  # keep yyyy/MM/
                       F.lpad(F.substring("release_date", 9, 1), 2, "0")  # pad day
                   ))
             .otherwise(F.col("release_date"))
        )
        .withColumn("release_date_dt", F.to_date("release_norm", "yyyy/MM/dd"))
        .withColumn("release_year", F.year("release_date_dt"))
        .withColumn("release_month", F.month("release_date_dt"))
        .drop("release_norm")
)

In [0]:
# sanity check
fact_games.select("release_date", "release_date_dt").distinct().show(100, False)


+------------+---------------+
|release_date|release_date_dt|
+------------+---------------+
|2019/09/9   |2019-09-09     |
|2020/12/30  |2020-12-30     |
|2022/02/14  |2022-02-14     |
|2019/01/31  |2019-01-31     |
|2019/01/26  |2019-01-26     |
|2021/09/28  |2021-09-28     |
|2021/05/13  |2021-05-13     |
|2019/02/21  |2019-02-21     |
|2019/04/4   |2019-04-04     |
|2019/05/17  |2019-05-17     |
|2019/11/27  |2019-11-27     |
|2019/07/2   |2019-07-02     |
|2012/12/19  |2012-12-19     |
|2020/12/4   |2020-12-04     |
|2011/10/11  |2011-10-11     |
|2020/04/22  |2020-04-22     |
|2019/01/20  |2019-01-20     |
|2019/10/8   |2019-10-08     |
|2019/10/19  |2019-10-19     |
|2021/05/31  |2021-05-31     |
|2020/09/22  |2020-09-22     |
|2019/07/25  |2019-07-25     |
|2020/12/11  |2020-12-11     |
|2020/05/23  |2020-05-23     |
|2019/11/13  |2019-11-13     |
|2019/06/20  |2019-06-20     |
|2019/12/27  |2019-12-27     |
|2019/09/23  |2019-09-23     |
|2009/08/6   |2009-08-06     |
|2019/01

## Price columns (prices are in cents)

In [0]:
fact_games = (
    fact_games
      .withColumn("price_cents", F.col("initialprice").cast("long"))
      .withColumn("price_eur",   F.when(F.col("price_cents").isNotNull(), F.col("price_cents")/100.0).otherwise(0.0))
      .withColumn("discount_pct", F.col("discount").cast("double"))
      .withColumn("price_after_discount", F.round(F.col("price_eur") * (1 - F.coalesce(F.col("discount_pct"), F.lit(0.0))/100.0), 2))
      .withColumn("is_discounted", F.coalesce(F.col("discount_pct"), F.lit(0.0)) > 0)
)

## REQUIRED_AGE

In [0]:
fact_games = (
    fact_games
      .withColumn(
          "required_age_clean",
          F.regexp_extract(F.coalesce(F.col("required_age").cast("string"), F.lit("")), r"(\d+)", 1).cast("int")
      )
      .withColumn(
          "required_age_clean",
          F.when((F.col("required_age_clean") >= 0) & (F.col("required_age_clean") <= 21),
                 F.col("required_age_clean"))
      )
)


In [0]:
# sanity check
fact_games.groupBy("required_age_clean").count().orderBy("required_age_clean").show(30, False)

+------------------+-----+
|required_age_clean|count|
+------------------+-----+
|NULL              |5    |
|0                 |55030|
|3                 |3    |
|5                 |1    |
|6                 |4    |
|7                 |3    |
|8                 |3    |
|9                 |1    |
|10                |7    |
|12                |32   |
|13                |26   |
|14                |10   |
|15                |265  |
|16                |38   |
|17                |38   |
|18                |223  |
|20                |1    |
|21                |1    |
+------------------+-----+



## CATEGORIES

In [0]:
# Categories : explode them to create a bridge table

bridge_game_category = (
    df.select(
            F.col("data.appid"),
            F.explode_outer("data.categories").alias("category_raw")
        )
        .where(F.col("category_raw").isNotNull())
        .withColumn("category_name", F.trim(F.col("category_raw")))
        .select("appid", "category_name")
        .dropDuplicates()
)

bridge_game_category.show(5)

+-------+--------------------+
|  appid|       category_name|
+-------+--------------------+
|1013600|       Single-player|
|1017160|  Steam Achievements|
|1025430|Shared/Split Scre...|
|1029750|       Single-player|
|1032080|       Single-player|
+-------+--------------------+
only showing top 5 rows


In [0]:
# 1) Grab all field names inside the tags struct
tag_fields = df.schema["data"].dataType["tags"].dataType.names

# 2) Build an array of structs: [{tag: "<name>", val: <value>}, ...]
pairs = F.array(*[
    F.struct(F.lit(k).alias("tag"), F.col(f"data.tags.`{k}`").alias("val"))
    for k in tag_fields
])

# 3) Explode to bridge (appid ↔ tag), keep only positives
bridge_game_tags = (
    df
    .select(F.col("data.appid").alias("appid"), F.explode_outer(pairs).alias("kv"))
    .select("appid", F.col("kv.tag").alias("tag"), F.col("kv.val").alias("value"))
    .where(F.col("value").cast("long") > 0)
    .select("appid", "tag")
    .dropDuplicates(["appid", "tag"])
)

bridge_game_tags.show(10, truncate=False)

+-------+-------------+
|appid  |tag          |
+-------+-------------+
|1000010|Fantasy      |
|1000000|Side Scroller|
|10     |Survival     |
|1000010|Perma Death  |
|1000040|Simulation   |
|1000130|Strategy     |
|1000080|Rogue-lite   |
|1000080|Linear       |
|1000130|Casual       |
|1000280|Anime        |
+-------+-------------+
only showing top 10 rows


## PLATFORMS

In [0]:
# 1) List all fields currently present under data.platforms (linux/mac/windows/...).
platform_fields = [f.name for f in df.schema["data"].dataType["platforms"].dataType.fields]

# 2) Make an array of {platform, supported} structs, then explode it.
platform_pairs = F.array(*[
    F.struct(
        F.lit(k).alias("platform_raw"),
        F.col(f"data.platforms.`{k}`").cast("boolean").alias("supported")
    )
    for k in platform_fields
])

bridge_game_platform = (
    df
    .select(F.col("data.appid").alias("appid"), F.explode_outer(platform_pairs).alias("kv"))
    .select("appid",
            F.col("kv.platform_raw").alias("platform_raw"),
            F.col("kv.supported").alias("supported"))
    .where(F.col("supported") == True)                # keep only supported platforms
    .withColumn("platform",
                F.initcap(F.regexp_replace(F.col("platform_raw"), r"_", " ")))  # e.g., steam_deck -> Steam Deck
    .select("appid", "platform")
    .dropDuplicates()
)

# Optional: small dimension of platforms
dim_platform = (
    bridge_game_platform
    .select(F.col("platform").alias("platform_name"))
    .dropDuplicates()
    .orderBy("platform_name")
)

# Quick peek
bridge_game_platform.show(10, truncate=False)
dim_platform.show(truncate=False)

+-------+--------+
|appid  |platform|
+-------+--------+
|1002270|Windows |
|1018800|Windows |
|1014900|Windows |
|1014310|Mac     |
|1027810|Windows |
|1022010|Windows |
|1030210|Windows |
|1037570|Windows |
|1007830|Mac     |
|1029870|Windows |
+-------+--------+
only showing top 10 rows
+-------------+
|platform_name|
+-------------+
|Linux        |
|Mac          |
|Windows      |
+-------------+



In [0]:
# Keep the fact skinny
fact_games = fact_games.drop("languages_text")
fact_games = fact_games.drop("publisher_name")
fact_games = fact_games.drop("genre")
fact_games = fact_games.drop("owners_text")
fact_games = fact_games.drop("release_date")
fact_games = fact_games.drop("required_age")
fact_games = fact_games.drop("type")
fact_games = fact_games.drop("short_description")
fact_games = fact_games.drop("initialprice")

In [0]:
fact_games.columns

['appid',
 'ccu',
 'developer_name',
 'discount',
 'header_image',
 'price',
 'name',
 'positive',
 'negative',
 'website',
 'total_reviews',
 'positive_rate',
 'owners_label',
 'owners_min',
 'release_date_dt',
 'release_year',
 'release_month',
 'price_cents',
 'price_eur',
 'discount_pct',
 'price_after_discount',
 'is_discounted',
 'required_age_clean']

In [0]:
# --- Save all fact & bridge tables into the "steam" database ---
spark.sql("CREATE DATABASE IF NOT EXISTS steam")
spark.sql("USE steam")

# --- Write all tables as Delta ---
(
    fact_games.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("mergeSchema", "true")
    .saveAsTable("steam.fact_games")
)

(
    bridge_game_genre.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("mergeSchema", "true")
    .saveAsTable("steam.bridge_game_genre")
)

(
    bridge_game_language.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("mergeSchema", "true")
    .saveAsTable("steam.bridge_game_language")
)

(
    bridge_game_category.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("mergeSchema", "true")
    .saveAsTable("steam.bridge_game_category")
)

(
    bridge_game_publisher.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("mergeSchema", "true")
    .saveAsTable("steam.bridge_game_publisher")
)

(
    bridge_game_tags.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("mergeSchema", "true")
    .saveAsTable("steam.bridge_game_tags")
)

(bridge_game_platform.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").option("mergeSchema", "true")
 .saveAsTable("steam.bridge_game_platform"))

# --- Quick sanity check ---
for t in [
    "fact_games",
    "bridge_game_genre",
    "bridge_game_language",
    "bridge_game_category",
    "bridge_game_publisher",
    "bridge_game_tags",
    "bridge_game_platform"
]:
    print(f"{t}: {spark.table(f'steam.{t}').count()} rows")


fact_games: 55691 rows
bridge_game_genre: 157110 rows
bridge_game_language: 196593 rows
bridge_game_category: 191255 rows
bridge_game_publisher: 56978 rows
bridge_game_tags: 675491 rows
bridge_game_platform: 76904 rows


In [0]:
# Create dimension tables

from pyspark.sql import functions as F

# Ensure schema exists and is selected
spark.sql("CREATE DATABASE IF NOT EXISTS steam")
spark.sql("USE steam")

# 1) Build dimension DataFrames (distinct, sorted for readability)
dim_genre     = bridge_game_genre.select(F.col("genre").alias("genre_name")).dropDuplicates().orderBy("genre_name")
dim_language  = bridge_game_language.select(F.col("language").alias("language_name")).dropDuplicates().orderBy("language_name")
dim_category  = bridge_game_category.select(F.col("category_name")).dropDuplicates().orderBy("category_name")
dim_publisher = bridge_game_publisher.select(F.col("publisher").alias("publisher_name")).dropDuplicates().orderBy("publisher_name")
dim_tag       = bridge_game_tags.select(F.col("tag").alias("tag_name")).dropDuplicates().orderBy("tag_name")
dim_platform  = bridge_game_platform.select(F.col("platform").alias("platform_name")).dropDuplicates().orderBy("platform_name")

# 2) Save dimensions as Delta tables (create or replace)
def save_dim(df, full_name):
    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .option("mergeSchema", "true")
       .saveAsTable(full_name))

save_dim(dim_genre,     "steam.dim_genre")
save_dim(dim_language,  "steam.dim_language")
save_dim(dim_category,  "steam.dim_category")
save_dim(dim_publisher, "steam.dim_publisher")
save_dim(dim_tag,       "steam.dim_tag")
save_dim(dim_platform,  "steam.dim_platform")

# 3) Quick sanity counts
for t in ["dim_genre","dim_language","dim_category","dim_publisher","dim_tag","dim_platform"]:
    cnt = spark.table(f"steam.{t}").count()
    print(f"{t}: {cnt} rows")


dim_genre: 28 rows
dim_language: 69 rows
dim_category: 36 rows
dim_publisher: 29072 rows
dim_tag: 441 rows
dim_platform: 3 rows
